# Cost Guards and Circuit Breakers

Agents are **non-deterministic spend engines**. A single user message can trigger five LLM calls, twelve tool invocations, and thousands of tokens — or loop forever retrying a failed check. **Cost guards** enforce budgets *before* the bill arrives. **Circuit breakers** stop runaway behavior *during* execution and pause traffic *after* repeated failures.

```
  Request ──► Budget check ──► Agent loop ──► per-step guard ──► answer
                  │                │
                  │                └── circuit breaker trips → fallback
                  └── daily cap exceeded → 429 / escalate
```

| Guard type | Scope | Trip condition |
|------------|-------|----------------|
| **Per-run budget** | One trace | `cost > $0.05` or `steps > 8` |
| **Token ceiling** | One LLM chain | `tokens > 4000` |
| **Loop detector** | Tool sequence | Same tool ≥ 3 times |
| **Circuit breaker** | Service / tenant | N trips in window → OPEN |
| **Daily ledger** | Tenant / team | Cumulative spend cap |

This notebook builds a complete guard stack in plain Python for **ShopCo Support Hub** and **ReleaseFlow**: budget ledgers, per-step enforcement, circuit breaker states (`CLOSED` → `OPEN` → `HALF_OPEN`), graceful fallbacks, and trip observability.

In [ ]:
"""
Cost guards and circuit breakers — ShopCo + ReleaseFlow.
"""

import json
import os
import re
import uuid
from collections import Counter, deque
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
TOKEN_COST_PER_1K = 0.002


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days with receipt. [pol-returns-01]",
    "shipping": "Free shipping on orders over $50. [pol-shipping-02]",
    "hours": "Support hours Mon–Fri 9am–6pm ET. [faq-hours-05]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

RELEASE_CHECKS = {
    "unit_tests": {"status": "pass"},
    "security_scan": {"status": "warn"},
    "changelog": {"status": "pass"},
}

print("Cost guards lab ready.")

---

## 1. Why Agents Need Cost Guards — Not Just Rate Limits

Traditional APIs charge per request. Agents charge per **internal step**:

- A ReAct loop may call `search_policy` six times before answering.
- A reflection pattern doubles LLM calls per user turn.
- A ReleaseFlow agent may retry `security_scan` until the context window fills.

**Rate limits** cap requests per minute. **Cost guards** cap tokens, dollars, and steps per run — and aggregate spend per tenant per day. Without both, one abusive session or one buggy prompt can burn a monthly budget in hours.

---

## 2. Guard Configuration — Tiers by Environment

In [ ]:
@dataclass(frozen=True)
class CostGuardConfig:
    max_steps: int = 8
    max_tokens_per_run: int = 4000
    max_cost_per_run_usd: float = 0.05
    max_tool_repeats: int = 3
    daily_budget_usd: float = 5.00
    circuit_trip_threshold: int = 3
    circuit_reset_after_sec: int = 60


GUARD_TIERS: dict[str, CostGuardConfig] = {
    "dev": CostGuardConfig(max_steps=20, max_cost_per_run_usd=0.20, daily_budget_usd=50.0),
    "staging": CostGuardConfig(max_steps=12, max_cost_per_run_usd=0.08, daily_budget_usd=20.0),
    "prod": CostGuardConfig(max_steps=8, max_cost_per_run_usd=0.05, daily_budget_usd=5.0),
}

ACTIVE_CONFIG = GUARD_TIERS["prod"]
print(f"Prod guard: max_steps={ACTIVE_CONFIG.max_steps}, max_cost=${ACTIVE_CONFIG.max_cost_per_run_usd}")

---

## 3. Trace Model — Track Tokens and Cost Per Step

In [ ]:
class SpanKind(str, Enum):
    AGENT = "agent"
    LLM = "llm"
    TOOL = "tool"
    ROUTER = "router"


@dataclass
class Span:
    name: str
    kind: SpanKind
    tokens: int = 0
    cost_usd: float = 0.0

    def __post_init__(self) -> None:
        if self.tokens and not self.cost_usd:
            self.cost_usd = round(self.tokens / 1000 * TOKEN_COST_PER_1K, 5)


@dataclass
class AgentRunTrace:
    trace_id: str
    service: str
    tenant_id: str
    user_message: str
    spans: list[Span] = field(default_factory=list)
    final_answer: str = ""
    terminated_reason: str = "complete"
    guard_trips: list[str] = field(default_factory=list)

    def add_span(self, name: str, kind: SpanKind, tokens: int = 0) -> Span:
        span = Span(name, kind, tokens)
        self.spans.append(span)
        return span

    def total_tokens(self) -> int:
        return sum(s.tokens for s in self.spans)

    def cost_usd(self) -> float:
        return round(sum(s.cost_usd for s in self.spans), 5)

    def tool_names(self) -> list[str]:
        return [s.name for s in self.spans if s.kind == SpanKind.TOOL]

---

## 4. Budget Ledger — Per-Tenant Daily Spend

In [ ]:
@dataclass
class LedgerEntry:
    tenant_id: str
    trace_id: str
    cost_usd: float
    ts: str = field(default_factory=utc_now)


class BudgetLedger:
    def __init__(self, daily_cap_usd: float) -> None:
        self.daily_cap_usd = daily_cap_usd
        self.entries: list[LedgerEntry] = []

    def spend(self, tenant_id: str, trace_id: str, cost_usd: float) -> None:
        self.entries.append(LedgerEntry(tenant_id, trace_id, cost_usd))

    def tenant_total(self, tenant_id: str) -> float:
        return round(sum(e.cost_usd for e in self.entries if e.tenant_id == tenant_id), 4)

    def can_afford(self, tenant_id: str, estimated_cost: float = 0.01) -> tuple[bool, str]:
        projected = self.tenant_total(tenant_id) + estimated_cost
        if projected > self.daily_cap_usd:
            return False, f"daily budget ${self.daily_cap_usd} exceeded (at ${self.tenant_total(tenant_id)})"
        return True, "ok"


ledger = BudgetLedger(ACTIVE_CONFIG.daily_budget_usd)
ledger.spend("shopco-tenant-a", "tr-001", 0.02)
ledger.spend("shopco-tenant-a", "tr-002", 0.03)
print("Tenant total:", ledger.tenant_total("shopco-tenant-a"))
print("Can afford next run:", ledger.can_afford("shopco-tenant-a", 0.01))

---

## 5. Per-Run Guards — Step, Token, Cost, Loop

In [ ]:
@dataclass
class GuardResult:
    allowed: bool
    trip_reason: str | None = None
    guard_name: str | None = None


def detect_tool_loop(trace: AgentRunTrace, threshold: int) -> GuardResult:
    counts = Counter(trace.tool_names())
    for name, cnt in counts.items():
        if cnt >= threshold:
            return GuardResult(False, f"tool_loop:{name}x{cnt}", "loop_detector")
    return GuardResult(True)


def check_run_guards(trace: AgentRunTrace, config: CostGuardConfig) -> GuardResult:
    if len(trace.spans) >= config.max_steps:
        return GuardResult(False, f"max_steps:{config.max_steps}", "step_guard")
    if trace.total_tokens() > config.max_tokens_per_run:
        return GuardResult(False, f"max_tokens:{trace.total_tokens()}", "token_guard")
    if trace.cost_usd() > config.max_cost_per_run_usd:
        return GuardResult(False, f"max_cost:${trace.cost_usd()}", "cost_guard")
    loop = detect_tool_loop(trace, config.max_tool_repeats)
    if not loop.allowed:
        return loop
    return GuardResult(True)

---

## 6. Circuit Breaker — CLOSED, OPEN, HALF_OPEN

Per-run guards stop **one bad trace**. A **circuit breaker** stops **many bad traces** from hammering downstream services or draining budget.

In [ ]:
class BreakerState(str, Enum):
    CLOSED = "closed"      # normal traffic
    OPEN = "open"          # reject / fallback
    HALF_OPEN = "half_open"  # probe with limited traffic


@dataclass
class BreakerTrip:
    trace_id: str
    reason: str
    ts: str = field(default_factory=utc_now)


class ServiceCircuitBreaker:
    def __init__(self, service: str, config: CostGuardConfig) -> None:
        self.service = service
        self.config = config
        self.state = BreakerState.CLOSED
        self.trips: deque[BreakerTrip] = deque(maxlen=50)
        self.last_state_change: str = utc_now()
        self.half_open_probes: int = 0

    def record_trip(self, trace_id: str, reason: str) -> None:
        self.trips.append(BreakerTrip(trace_id, reason))
        recent = [t for t in self.trips][-self.config.circuit_trip_threshold:]
        if len(recent) >= self.config.circuit_trip_threshold and self.state == BreakerState.CLOSED:
            self._transition(BreakerState.OPEN)

    def _transition(self, new_state: BreakerState) -> None:
        self.state = new_state
        self.last_state_change = utc_now()
        if new_state == BreakerState.HALF_OPEN:
            self.half_open_probes = 0

    def allow_request(self) -> GuardResult:
        if self.state == BreakerState.CLOSED:
            return GuardResult(True)
        if self.state == BreakerState.OPEN:
            elapsed = (datetime.fromisoformat(utc_now()) - datetime.fromisoformat(self.last_state_change)).total_seconds()
            if elapsed >= self.config.circuit_reset_after_sec:
                self._transition(BreakerState.HALF_OPEN)
                return GuardResult(True)
            return GuardResult(False, "circuit_open", "circuit_breaker")
        # HALF_OPEN: allow limited probes
        if self.half_open_probes >= 2:
            return GuardResult(False, "half_open_probe_limit", "circuit_breaker")
        self.half_open_probes += 1
        return GuardResult(True)

    def record_success(self) -> None:
        if self.state == BreakerState.HALF_OPEN:
            self._transition(BreakerState.CLOSED)
            self.trips.clear()

    def record_failure(self, trace_id: str, reason: str) -> None:
        self.record_trip(trace_id, reason)
        if self.state == BreakerState.HALF_OPEN:
            self._transition(BreakerState.OPEN)


shopco_breaker = ServiceCircuitBreaker("shopco-support", ACTIVE_CONFIG)
print(f"Initial state: {shopco_breaker.state.value}")

---

## 7. Fallback Responses — Graceful Degradation

In [ ]:
FALLBACKS: dict[str, str] = {
    "circuit_open": "Our support agent is temporarily unavailable. Please email support@shopco.example or try again in a few minutes.",
    "daily_budget": "You've reached today's automated support limit. A human agent will follow up shortly.",
    "max_steps": "I couldn't complete your request within safe limits. Here's our returns policy: Returns within 30 days with receipt. [pol-returns-01]",
    "tool_loop": "I got stuck searching — here's a direct link to our help center: help.shopco.example/returns",
    "max_cost": "This request was too complex for automated handling. Escalating to a human agent.",
}


def fallback_for(trip_reason: str | None) -> str:
    if not trip_reason:
        return "Something went wrong. Please try again."
    key = trip_reason.split(":")[0]
    if key.startswith("tool_loop"):
        key = "tool_loop"
    if key.startswith("max_steps"):
        key = "max_steps"
    if key.startswith("max_cost"):
        key = "max_cost"
    if key == "circuit_open":
        key = "circuit_open"
    return FALLBACKS.get(key, FALLBACKS["max_steps"])

---

## 8. Guarded Executor — Wrap Any Agent

In [ ]:
@dataclass
class GuardedRunResult:
    trace: AgentRunTrace
    success: bool
    blocked_by: str | None = None


class GuardedExecutor:
    def __init__(
        self,
        service: str,
        config: CostGuardConfig,
        ledger: BudgetLedger,
        breaker: ServiceCircuitBreaker,
    ) -> None:
        self.service = service
        self.config = config
        self.ledger = ledger
        self.breaker = breaker

    def pre_check(self, tenant_id: str) -> GuardResult:
        breaker_ok = self.breaker.allow_request()
        if not breaker_ok.allowed:
            return breaker_ok
        afford_ok, reason = self.ledger.can_afford(tenant_id)
        if not afford_ok:
            return GuardResult(False, reason, "daily_budget")
        return GuardResult(True)

    def post_step_check(self, trace: AgentRunTrace) -> GuardResult:
        return check_run_guards(trace, self.config)

    def finalize(self, trace: AgentRunTrace, guard: GuardResult) -> GuardedRunResult:
        if guard.allowed:
            self.ledger.spend(trace.tenant_id, trace.trace_id, trace.cost_usd())
            self.breaker.record_success()
            return GuardedRunResult(trace, True)
        trace.guard_trips.append(guard.trip_reason or "unknown")
        trace.terminated_reason = guard.trip_reason or "guard_trip"
        trace.final_answer = fallback_for(guard.trip_reason)
        self.breaker.record_failure(trace.trace_id, guard.trip_reason or "failure")
        return GuardedRunResult(trace, False, guard.guard_name)


executor = GuardedExecutor("shopco-support", ACTIVE_CONFIG, ledger, shopco_breaker)
print("GuardedExecutor ready.")

---

## 9. ShopCo Agent — Healthy vs Looping Runs

In [ ]:
def shopco_healthy_run(message: str, tenant_id: str) -> AgentRunTrace:
    trace = AgentRunTrace(f"tr-{uuid.uuid4().hex[:6]}", "shopco-support", tenant_id, message)
    trace.add_span("route_intent", SpanKind.ROUTER, 80)
    trace.add_span("search_policy", SpanKind.TOOL, 120)
    trace.add_span("compose_answer", SpanKind.LLM, 200)
    trace.final_answer = POLICY_SNIPPETS["returns"] if "return" in message.lower() else POLICY_SNIPPETS["shipping"]
    return trace


def shopco_looping_run(message: str, tenant_id: str, repeats: int = 5) -> AgentRunTrace:
    trace = AgentRunTrace(f"tr-{uuid.uuid4().hex[:6]}", "shopco-support", tenant_id, message)
    trace.add_span("route_intent", SpanKind.ROUTER, 90)
    for _ in range(repeats):
        trace.add_span("search_policy", SpanKind.TOOL, 300)
    return trace


def run_with_guards(
    executor: GuardedExecutor,
    build_trace: Callable[[], AgentRunTrace],
) -> GuardedRunResult:
    trace = build_trace()
    pre = executor.pre_check(trace.tenant_id)
    if not pre.allowed:
        trip_key = pre.guard_name if pre.guard_name == "daily_budget" else pre.trip_reason
        trace.final_answer = fallback_for(trip_key)
        trace.terminated_reason = pre.trip_reason or "pre_check"
        return GuardedRunResult(trace, False, pre.guard_name)
    for _ in range(executor.config.max_steps + 2):
        guard = executor.post_step_check(trace)
        if not guard.allowed:
            return executor.finalize(trace, guard)
        if trace.final_answer:
            return executor.finalize(trace, GuardResult(True))
        trace.add_span("search_policy", SpanKind.TOOL, 250)
    return executor.finalize(trace, GuardResult(False, "max_steps:exhausted", "step_guard"))


healthy = run_with_guards(executor, lambda: shopco_healthy_run("return policy?", "shopco-tenant-a"))
print("Healthy:", healthy.success, healthy.trace.final_answer[:50])

---

## 10. Demo — Loop Trip and Circuit OPEN

In [ ]:
demo_breaker = ServiceCircuitBreaker("shopco-support", ACTIVE_CONFIG)
demo_ledger = BudgetLedger(ACTIVE_CONFIG.daily_budget_usd)
demo_executor = GuardedExecutor("shopco-support", ACTIVE_CONFIG, demo_ledger, demo_breaker)

loop_results: list[GuardedRunResult] = []
for i in range(4):
    result = run_with_guards(
        demo_executor,
        lambda: shopco_looping_run("where is my order?", "shopco-tenant-b", repeats=4),
    )
    loop_results.append(result)
    print(f"Run {i+1}: success={result.success}, breaker={demo_breaker.state.value}, blocked_by={result.blocked_by}")

blocked = run_with_guards(demo_executor, lambda: shopco_healthy_run("returns?", "shopco-tenant-b"))
print(f"After trips — circuit blocks healthy request: {not blocked.success}, answer preview: {blocked.trace.final_answer[:60]}")

---

## 11. Cost Overrun — Expensive Reflection Chain

In [ ]:
def shopco_expensive_run(message: str, tenant_id: str, n_reflections: int = 12) -> AgentRunTrace:
    trace = AgentRunTrace(f"tr-{uuid.uuid4().hex[:6]}", "shopco-support", tenant_id, message)
    trace.add_span("route_intent", SpanKind.ROUTER, 100)
    for i in range(n_reflections):
        trace.add_span(f"reflect_{i}", SpanKind.LLM, 450)
    trace.final_answer = "Over-refined answer with no new information."
    return trace


cost_breaker = ServiceCircuitBreaker("shopco-support", ACTIVE_CONFIG)
cost_ledger = BudgetLedger(ACTIVE_CONFIG.daily_budget_usd)
cost_executor = GuardedExecutor("shopco-support", ACTIVE_CONFIG, cost_ledger, cost_breaker)

expensive_result = run_with_guards(
    cost_executor,
    lambda: shopco_expensive_run("complex billing dispute", "shopco-tenant-c"),
)
print(f"Tokens: {expensive_result.trace.total_tokens()}, cost: ${expensive_result.trace.cost_usd()}")
print(f"Tripped: {expensive_result.trace.guard_trips}, blocked_by: {expensive_result.blocked_by}")

---

## 12. Daily Budget Exhaustion

In [ ]:
tight_ledger = BudgetLedger(daily_cap_usd=0.06)
tight_breaker = ServiceCircuitBreaker("shopco-support", ACTIVE_CONFIG)
tight_executor = GuardedExecutor("shopco-support", ACTIVE_CONFIG, tight_ledger, tight_breaker)

for i in range(3):
    r = run_with_guards(tight_executor, lambda: shopco_healthy_run(f"q{i}", "shopco-tenant-d"))
    print(f"Run {i+1}: success={r.success}, tenant_spend=${tight_ledger.tenant_total('shopco-tenant-d')}")

over_budget = tight_executor.pre_check("shopco-tenant-d")
print("Pre-check after budget exhausted:", over_budget)

---

## 13. ReleaseFlow — Retry Storm Guard

In [ ]:
def releaseflow_retry_storm(release_id: str, retries: int = 6) -> AgentRunTrace:
    trace = AgentRunTrace(f"tr-rf-{uuid.uuid4().hex[:6]}", "releaseflow", "releaseflow-team", f"deploy {release_id}")
    trace.add_span("plan_release", SpanKind.ROUTER, 60)
    for i in range(retries):
        trace.add_span("run_security_scan", SpanKind.TOOL, 40)
        trace.add_span("llm_analyze_failure", SpanKind.LLM, 500)
    return trace


RF_CONFIG = CostGuardConfig(max_steps=10, max_cost_per_run_usd=0.03, max_tool_repeats=3, daily_budget_usd=2.0)
rf_breaker = ServiceCircuitBreaker("releaseflow", RF_CONFIG)
rf_ledger = BudgetLedger(RF_CONFIG.daily_budget_usd)
rf_executor = GuardedExecutor("releaseflow", RF_CONFIG, rf_ledger, rf_breaker)

rf_result = run_with_guards(
    rf_executor,
    lambda: releaseflow_retry_storm("2.4.0", retries=5),
)
print(f"ReleaseFlow blocked: {not rf_result.success}")
print(f"Reason: {rf_result.trace.terminated_reason}")
print(f"Fallback: {rf_result.trace.final_answer[:80]}")

---

## 14. Trip Observability — Metrics and Alerts

In [ ]:
@dataclass
class GuardMetrics:
    service: str
    total_runs: int
    blocked_runs: int
    trips_by_guard: dict[str, int]
    breaker_state: str
    avg_cost_usd: float


def aggregate_guard_metrics(results: list[GuardedRunResult], breaker: ServiceCircuitBreaker) -> GuardMetrics:
    blocked = [r for r in results if not r.success]
    trips: Counter[str] = Counter()
    for r in blocked:
        trips[r.blocked_by or "unknown"] += 1
    costs = [r.trace.cost_usd() for r in results]
    return GuardMetrics(
        service=breaker.service,
        total_runs=len(results),
        blocked_runs=len(blocked),
        trips_by_guard=dict(trips),
        breaker_state=breaker.state.value,
        avg_cost_usd=round(sum(costs) / len(costs), 4) if costs else 0.0,
    )


ALERT_RULES = {
    "block_rate_max": 0.30,
    "breaker_open": "open",
}


def evaluate_guard_alerts(metrics: GuardMetrics) -> list[str]:
    alerts: list[str] = []
    block_rate = metrics.blocked_runs / metrics.total_runs if metrics.total_runs else 0
    if block_rate > ALERT_RULES["block_rate_max"]:
        alerts.append(f"block_rate {block_rate:.0%} exceeds {ALERT_RULES['block_rate_max']:.0%}")
    if metrics.breaker_state == ALERT_RULES["breaker_open"]:
        alerts.append(f"circuit breaker OPEN on {metrics.service}")
    return alerts


all_results = loop_results + [expensive_result, rf_result]
metrics = aggregate_guard_metrics(all_results, demo_breaker)
print(pretty(metrics.__dict__))
print("Alerts:", evaluate_guard_alerts(metrics))

---

## 15. HALF_OPEN Recovery — Probe Then Close

In [ ]:
def simulate_recovery(breaker: ServiceCircuitBreaker, executor: GuardedExecutor) -> list[str]:
    log: list[str] = []
    log.append(f"start: {breaker.state.value}")
    breaker._transition(BreakerState.HALF_OPEN)
    log.append(f"forced half_open: {breaker.state.value}")
    ok_run = run_with_guards(executor, lambda: shopco_healthy_run("hours?", "shopco-tenant-b"))
    log.append(f"probe success={ok_run.success}, state={breaker.state.value}")
    return log


recovery_log = simulate_recovery(demo_breaker, demo_executor)
for line in recovery_log:
    print(line)

---

## 16. Guard Policy — When to Escalate vs Fallback

In [ ]:
@dataclass
class EscalationPolicy:
    escalate_on: set[str] = field(default_factory=lambda: {"max_cost", "daily_budget"})
    fallback_on: set[str] = field(default_factory=lambda: {"tool_loop", "max_steps", "circuit_open"})


def route_after_trip(trip_reason: str | None, policy: EscalationPolicy) -> str:
    key = (trip_reason or "").split(":")[0]
    if key.startswith("tool_loop"):
        key = "tool_loop"
    if key.startswith("max_steps"):
        key = "max_steps"
    if key.startswith("max_cost"):
        key = "max_cost"
    if key in policy.escalate_on:
        return "escalate_human"
    if key in policy.fallback_on:
        return "fallback_message"
    return "retry_later"


policy = EscalationPolicy()
for reason in ["tool_loop:search_policyx4", "max_cost:$0.08", "circuit_open"]:
    print(reason, "→", route_after_trip(reason, policy))

---

## 17. Unified Guard Stack — One Entry Point

In [ ]:
class GuardStack:
    def __init__(self, service: str, env: str = "prod") -> None:
        self.config = GUARD_TIERS[env]
        self.ledger = BudgetLedger(self.config.daily_budget_usd)
        self.breaker = ServiceCircuitBreaker(service, self.config)
        self.executor = GuardedExecutor(service, self.config, self.ledger, self.breaker)
        self.history: list[GuardedRunResult] = []

    def run(self, build_trace: Callable[[], AgentRunTrace]) -> GuardedRunResult:
        result = run_with_guards(self.executor, build_trace)
        self.history.append(result)
        return result

    def status(self) -> dict[str, Any]:
        m = aggregate_guard_metrics(self.history, self.breaker)
        return {
            "service": self.breaker.service,
            "breaker_state": self.breaker.state.value,
            "runs": m.total_runs,
            "block_rate": round(m.blocked_runs / m.total_runs, 2) if m.total_runs else 0,
            "trips_by_guard": m.trips_by_guard,
        }


stack = GuardStack("shopco-support", "prod")
stack.run(lambda: shopco_healthy_run("shipping cost?", "tenant-x"))
stack.run(lambda: shopco_looping_run("order status", "tenant-x", repeats=4))
print(pretty(stack.status()))

---

## 18. Optional Live LLM — Cost-Aware Wrapper

Set `USE_LIVE_LLM = True` to call `gpt-4o-mini` with a hard token cap. The wrapper records spend into the ledger.

In [ ]:
def cost_aware_llm_call(prompt: str, max_tokens: int = 150) -> dict[str, Any]:
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0, max_tokens=max_tokens)
    resp = llm.invoke([HumanMessage(content=prompt)])
    usage = getattr(resp, "usage_metadata", None) or {}
    tokens = usage.get("total_tokens", len(str(resp.content).split()) * 2)
    cost = round(tokens / 1000 * TOKEN_COST_PER_1K, 5)
    return {"answer": str(resp.content), "tokens": tokens, "cost_usd": cost}


if USE_LIVE_LLM:
    live = cost_aware_llm_call("Summarize ShopCo return policy in one sentence.")
    ledger.spend("shopco-tenant-a", "tr-live", live["cost_usd"])
    print(live)
else:
    print("Simulated LLM call: tokens=180, cost=$0.00036 — enable USE_LIVE_LLM for real spend tracking.")

---

## 19. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Guard only at API edge | Internal loops still burn tokens | Per-step check inside agent loop |
| No fallback message | User sees empty error | Map trip reason → helpful response |
| Circuit never resets | Permanent outage | HALF_OPEN probe after cooldown |
| Same limits dev and prod | Can't debug in dev | `GUARD_TIERS` by environment |
| Ledger per run not tenant | One user drains team budget | Aggregate by `tenant_id` |
| Trip without observability | Repeat incidents | `trips_by_guard` metrics + alerts |
| Escalate everything | Human queue overload | `EscalationPolicy` by trip type |

---

## 20. Quiz

1. What is the difference between a per-run cost guard and a circuit breaker?
2. Name the three circuit breaker states and what each allows.
3. What trip condition does `detect_tool_loop` check?
4. Why track spend in a `BudgetLedger` per tenant?
5. When should ReleaseFlow stop retrying `security_scan`?
6. What is the purpose of HALF_OPEN state?

---

## 21. Summary

Agent cost overruns and infinite loops are **production incidents**, not edge cases. Guard at three layers: **pre-request** (ledger + circuit state), **per-step** (steps, tokens, loops), and **post-trip** (fallback or escalate).

This notebook built `CostGuardConfig`, `BudgetLedger`, `check_run_guards`, `ServiceCircuitBreaker` with `CLOSED`/`OPEN`/`HALF_OPEN`, `GuardedExecutor`, ShopCo and ReleaseFlow demos, trip metrics, and `GuardStack` as a unified entry point.

Combine cost guards with the production quality loop: when `block_rate` spikes, treat it as a regression — fix the agent, not just the threshold.